# T Industries - semantic layer (run after the pipeline)
* **Metric view `t_hardware.t_sales_metrics`** (over `gold_sales_daily`, `gold_dim_product`, `gold_dim_customer`) - governed definitions of net sales, gross margin %, units, ASP and backorder rate (Finance / Operations business rules in SharePoint `Business Rules/`), with synonyms for Genie and TAI.
* **Customer photos** are no longer handled here: Genie One describes the uploaded photo and the *T Sales & Ops* agent matches the description to the catalog (see `04_setup_genie`). This notebook no longer creates the former `identify_product_photo` function.
* **Layer tags** - every object in `t_hardware` with a `bronze_` / `silver_` / `gold_` prefix gets the governed Unity Catalog tag `layer` = bronze / silver / gold (the only values the tag policy allows), so layers can be filtered in Catalog Explorer and used in ABAC policies.

In [0]:
dbutils.widgets.text("catalog", "mdk_demo_catalog")
dbutils.widgets.text("vision_model", "databricks-claude-sonnet-4-6")
C = dbutils.widgets.get("catalog")
VISION = dbutils.widgets.get("vision_model")
G = f"{C}.t_hardware"     # single schema: bronze_ / silver_ / gold_ / gov_ prefixes; semantic objects unprefixed

In [0]:
mv_yaml = f"""
version: 1.1
comment: >-
  T Industries governed sales metrics (CAD). Net sales = gross sales - discounts - returns, excluding freight and tax;
  cancelled orders excluded (Finance rule net_sales_definition.md). Backorder rate = backordered / ordered units
  (Operations rule backorder_definition.md).
source: {G}.gold_sales_daily
joins:
  - name: product
    source: {G}.gold_dim_product
    using: [sku]
    cardinality: many_to_one
  - name: customer
    source: {G}.gold_dim_customer
    using: [customer_id]
    cardinality: many_to_one
dimensions:
  - expr: order_date
  - name: order_month
    expr: DATE_TRUNC('MONTH', order_date)
    format:
      type: date
      date_format: locale_short_month
    synonyms: [month]
  - name: order_quarter
    expr: DATE_TRUNC('QUARTER', order_date)
    synonyms: [quarter, fiscal quarter]
  - name: order_year
    expr: YEAR(order_date)
    synonyms: [year]
  - expr: warehouse
    comment: Fulfilling warehouse - Delta BC or Toronto
    synonyms: [DC, distribution centre, branch]
  - expr: channel
    comment: Inside Sales, Field Sales, Web Portal or EDI
  - expr: sku
    synonyms: [item, part number, product code]
  - expr: product.product_name
  - expr: product.category
    comment: Locks, Levers, Door Closers, Hinges, Exit Devices, Faucets, Grab Bars, Bath Accessories
    synonyms: [product category]
  - expr: product.product_group
    comment: Door Hardware or Bathroom Fixtures
    synonyms: [division, business line]
  - expr: product.finish
  - expr: product.ansi_grade
    synonyms: [grade, BHMA grade]
  - expr: customer_id
  - expr: customer.customer_name
    synonyms: [customer, account]
  - expr: customer.segment
    comment: General Contractor, Condo Developer or Property Manager
    synonyms: [customer type]
  - expr: customer.region
    comment: Customer sales region - BC (British Columbia), ON (Ontario) or US
    synonyms: [province, territory]
  - expr: customer.account_manager
    synonyms: [sales rep, rep]
measures:
  - name: net_sales
    expr: SUM(net_sales_cad)
    comment: Gross sales - discounts - returns, CAD, excl. freight and tax
    format: {{type: currency, currency_code: CAD, abbreviation: compact}}
    synonyms: [revenue, sales, net revenue, turnover]
  - name: gross_sales
    expr: SUM(gross_sales_cad)
    comment: Shipped units x list price, CAD
    format: {{type: currency, currency_code: CAD, abbreviation: compact}}
  - name: discounts
    expr: SUM(discount_cad)
    format: {{type: currency, currency_code: CAD, abbreviation: compact}}
  - name: returns
    expr: SUM(returns_cad)
    format: {{type: currency, currency_code: CAD, abbreviation: compact}}
    synonyms: [credit memos, credits]
  - name: cogs
    expr: SUM(cogs_cad)
    format: {{type: currency, currency_code: CAD, abbreviation: compact}}
    synonyms: [cost of goods sold, cost]
  - name: gross_margin
    expr: SUM(gross_margin_cad)
    comment: Net sales - COGS, CAD
    format: {{type: currency, currency_code: CAD, abbreviation: compact}}
    synonyms: [margin, gross profit, GM]
  - name: gross_margin_pct
    expr: MEASURE(gross_margin) / NULLIF(MEASURE(net_sales), 0)
    comment: Gross margin / net sales
    format: {{type: percentage, decimal_places: {{type: exact, places: 1}}}}
    synonyms: [margin %, GM %, margin percent, profitability]
  - name: units_shipped
    expr: SUM(units_shipped)
    synonyms: [units, volume, quantity sold]
  - name: units_ordered
    expr: SUM(units_ordered)
  - name: units_backordered
    expr: SUM(units_backordered)
  - name: avg_selling_price
    expr: SUM(net_sales_cad) / NULLIF(SUM(units_shipped), 0)
    comment: Net sales / shipped units, CAD
    format: {{type: currency, currency_code: CAD, decimal_places: {{type: exact, places: 2}}}}
    synonyms: [ASP, average price, net price]
  - name: backorder_rate
    expr: SUM(units_backordered) / NULLIF(SUM(units_ordered), 0)
    comment: Backordered units / ordered units. Target below 5% per warehouse per month
    format: {{type: percentage, decimal_places: {{type: exact, places: 1}}}}
    synonyms: [backorders, fill rate gap, stockout rate]
  - name: customer_count
    expr: COUNT(DISTINCT customer_id)
    synonyms: [active customers, buying customers]
"""
spark.sql(f"CREATE OR REPLACE VIEW {G}.t_sales_metrics WITH METRICS LANGUAGE YAML AS $$\n{mv_yaml}\n$$")
display(spark.sql(f"""SELECT warehouse, MEASURE(net_sales) AS net_sales, MEASURE(gross_margin_pct) AS gm_pct, MEASURE(backorder_rate) AS bo_rate
                     FROM {G}.t_sales_metrics GROUP BY ALL"""))

In [0]:
# The workspace governed tag policy for `layer` only allows bronze / silver / gold, so gov_ and semantic objects stay untagged.
LAYER = {"bronze_": "bronze", "silver_": "silver", "gold_": "gold"}
SCHEMA = G.split(".")[1]
objs = spark.sql(f"SELECT table_name, table_type FROM {C}.information_schema.tables WHERE table_schema = '{SCHEMA}' AND table_name NOT LIKE '\\_\\_%' AND table_name NOT LIKE 'event_log%'").collect()
tagged, failed = [], []
for o in objs:
    layer = next((v for k, v in LAYER.items() if o.table_name.startswith(k)), None)
    if layer is None: continue
    kind = {"METRIC_VIEW": "VIEW", "VIEW": "VIEW", "MATERIALIZED_VIEW": "MATERIALIZED VIEW"}.get(o.table_type, "TABLE")
    try:
        spark.sql(f"ALTER {kind} {G}.`{o.table_name}` SET TAGS ('layer' = '{layer}')"); tagged.append((o.table_name, layer))
    except Exception as e:
        try:   # pipeline-owned materialized views: fall back to the generic SET TAG syntax
            spark.sql(f"SET TAG ON TABLE {G}.`{o.table_name}` layer = {layer}"); tagged.append((o.table_name, layer))
        except Exception as e2:
            failed.append((o.table_name, str(e2).splitlines()[0][:150]))
print(f"tagged {len(tagged)} objects; failed {len(failed)}")
for f in failed: print("  FAILED", f)
display(spark.sql(f"SELECT tag_value AS layer, count(*) AS objects FROM {C}.information_schema.table_tags WHERE schema_name = '{SCHEMA}' AND tag_name = 'layer' GROUP BY 1 ORDER BY 1"))